Datasets: https://www.kaggle.com/datasets/rohitudageri/credit-card-details/

**1. Setup**

In [26]:
# installing the dependencies
%pip install -q numpy pandas matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
# import the dependencies
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [28]:
# configurations
pd.set_option("display.max_columns", None)
sns.set_theme(style="darkgrid")

RANDOM_STATE = 42

**2. Load Datasets**

In [104]:
# read csv file into a pandas dataframe
df = pd.read_csv("D:\My Learning\ML projects\Credit-Card-Approval-Prediction\datasets\Credit_card.csv")
target = pd.read_csv("D:\My Learning\ML projects\Credit-Card-Approval-Prediction\datasets\Credit_card_label.csv")

**3. EDA**

In [105]:
df.head()

,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,NaN,-586,1,1,1,0,NaN,2
3,5009749,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2
4,5009752,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2


In [106]:
target.head()

,Ind_ID,label
0,5008827,1
1,5009744,1
2,5009746,1
3,5009749,1
4,5009752,1


In [107]:
df.shape

(1548, 18)

In [108]:
target.shape

(1548, 2)

In [109]:
# missing values - analysis
df.isnull().sum()

Ind_ID               0
GENDER               7
Car_Owner            0
Propert_Owner        0
CHILDREN             0
Annual_income       23
Type_Income          0
EDUCATION            0
Marital_status       0
Housing_type         0
Birthday_count      22
Employed_days        0
Mobile_phone         0
Work_Phone           0
Phone                0
EMAIL_ID             0
Type_Occupation    488
Family_Members       0
dtype: int64

In [110]:
# missing values - analysis
target.isnull().sum()

Ind_ID    0
label     0
dtype: int64

In [111]:
# identify the presence of encoded missing values
for col in df.columns:
    print(df[col].value_counts())

Ind_ID
5008827    1
5009744    1
5009746    1
5009749    1
5009752    1
          ..
5028645    1
5023655    1
5115992    1
5118219    1
5053790    1
Name: count, Length: 1548, dtype: int64
GENDER
F    973
M    568
Name: count, dtype: int64
Car_Owner
N    924
Y    624
Name: count, dtype: int64
Propert_Owner
Y    1010
N     538
Name: count, dtype: int64
CHILDREN
0     1091
1      305
2      134
3       16
4        1
14       1
Name: count, dtype: int64
Annual_income
135000.0    170
112500.0    144
180000.0    137
157500.0    125
225000.0    119
           ... 
165600.0      1
114750.0      1
47250.0       1
49500.0       1
69750.0       1
Name: count, Length: 115, dtype: int64
Type_Income
Working                 798
Commercial associate    365
Pensioner               269
State servant           116
Name: count, dtype: int64
EDUCATION
Secondary / secondary special    1031
Higher education                  426
Incomplete higher                  68
Lower secondary                    21
Aca

In [112]:
# identify the presence of encoded missing values
for col in target.columns:
    print(target[col].value_counts())

Ind_ID
5008827    1
5009744    1
5009746    1
5009749    1
5009752    1
          ..
5028645    1
5023655    1
5115992    1
5118219    1
5053790    1
Name: count, Length: 1548, dtype: int64
label
0    1373
1     175
Name: count, dtype: int64


In [113]:
# join both datafames using inner join
df = pd.merge(df, target, on='Ind_ID', how='inner')

In [114]:
df.head()

,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2,1
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,NaN,-586,1,1,1,0,NaN,2,1
3,5009749,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
4,5009752,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1


In [115]:
# identify duplicate rows
duplicate_mask = df.duplicated()
num_duplicates = duplicate_mask.sum()

print("Number of duplicate rows:", num_duplicates)
print("Duplicated row:\n",df[duplicate_mask])

# If you want to remove duplicates:
# df_no_duplicates = df.drop_duplicates()
# print("Shape after dropping duplicates:", df_no_duplicates.shape)

Number of duplicate rows: 0
Duplicated row:
 Empty DataFrame
Columns: [Ind_ID, GENDER, Car_Owner, Propert_Owner, CHILDREN, Annual_income, Type_Income, EDUCATION, Marital_status, Housing_type, Birthday_count, Employed_days, Mobile_phone, Work_Phone, Phone, EMAIL_ID, Type_Occupation, Family_Members, label]
Index: []


In [116]:
df = df.drop(columns=["Ind_ID"])

In [117]:
# class distribution
print(df["label"].value_counts())
print("-"*50)
print(df["label"].value_counts(normalize=True)*100)

label
0    1373
1     175
Name: count, dtype: int64
--------------------------------------------------
label
0    88.69509
1    11.30491
Name: proportion, dtype: float64


In [118]:
# Check constant columns
df.nunique()

GENDER                2
Car_Owner             2
Propert_Owner         2
CHILDREN              6
Annual_income       115
Type_Income           4
EDUCATION             5
Marital_status        5
Housing_type          6
Birthday_count     1270
Employed_days       956
Mobile_phone          1
Work_Phone            2
Phone                 2
EMAIL_ID              2
Type_Occupation      18
Family_Members        7
label                 2
dtype: int64

In [119]:
# Dropping constant column
df = df.drop(columns=["Mobile_phone"])

In [120]:
# Convert Birthday_count to age
df['Age'] = (-df['Birthday_count'] / 365.25).round(1)
df.drop('Birthday_count', axis=1, inplace=True)

In [121]:
df["Age"].dtype

dtype('float64')

In [122]:
df["Age"].isnull().sum()

np.int64(22)


Fill missing numerical values with the middle value (median) age, which is better if our data has extreme outliers

In [124]:
# Fill empty age cells with the median age
df['Age'] = df['Age'].fillna(df['Age'].median())

# For data with NO missing values
df['Age'] = df['Age'].astype(int)

In [125]:
# 365243 value means, individual is currently unemployed.
df['Is_Unemployed'] = (df['Employed_days'] == 365243).astype(int)

In [126]:
df.head()

,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Employed_days,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label,Age,Is_Unemployed
0,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,365243,0,0,0,NaN,2,1,51,1
1,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-586,1,1,0,NaN,2,1,37,0
2,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-586,1,1,0,NaN,2,1,42,0
3,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,-586,1,1,0,NaN,2,1,37,0
4,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-586,1,1,0,NaN,2,1,37,0


In [127]:
# Convert employed days to actual number of days
df['Employment_days'] = df['Employed_days'].abs()

# 365243 is a special value representing unemployment
df.loc[df['Employed_days'] == 365243, 'Employment_days'] = 0

df['Employment_years'] = (df['Employment_days'] / 365.25).round(1)

df.drop('Employed_days', axis=1, inplace=True)

In [128]:
df.head()

,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label,Age,Is_Unemployed,Employment_days,Employment_years
0,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,0,0,0,NaN,2,1,51,1,0,0.0
1,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37,0,586,1.6
2,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,42,0,586,1.6
3,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37,0,586,1.6
4,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37,0,586,1.6


In [129]:
# encode categorical column
df["GENDER"] = df["GENDER"].map({"M": 1, "F": 0})
df["Car_Owner"] = df["Car_Owner"].map({"Y": 1, "N": 0})
df["Propert_Owner"] = df["Propert_Owner"].map({"Y": 1, "N": 0})
df.head()

,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label,Age,Is_Unemployed,Employment_days,Employment_years
0,1.0,1,1,0,180000.0,Pensioner,Higher education,Married,House / apartment,0,0,0,NaN,2,1,51,1,0,0.0
1,0.0,1,0,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37,0,586,1.6
2,0.0,1,0,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,42,0,586,1.6
3,0.0,1,0,0,NaN,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37,0,586,1.6
4,0.0,1,0,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37,0,586,1.6


In [131]:
# missing values - analysis
df.isnull().sum()

GENDER                7
Car_Owner             0
Propert_Owner         0
CHILDREN              0
Annual_income        23
Type_Income           0
EDUCATION             0
Marital_status        0
Housing_type          0
Work_Phone            0
Phone                 0
EMAIL_ID              0
Type_Occupation     488
Family_Members        0
label                 0
Age                   0
Is_Unemployed         0
Employment_days       0
Employment_years      0
dtype: int64

In [134]:
df['Annual_income'] = df['Annual_income'].fillna(df['Annual_income'].median())

In [135]:
df.isnull().sum()


GENDER                7
Car_Owner             0
Propert_Owner         0
CHILDREN              0
Annual_income         0
Type_Income           0
EDUCATION             0
Marital_status        0
Housing_type          0
Work_Phone            0
Phone                 0
EMAIL_ID              0
Type_Occupation     488
Family_Members        0
label                 0
Age                   0
Is_Unemployed         0
Employment_days       0
Employment_years      0
dtype: int64